In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import pandas as pd
import numpy as np
import optuna
import gc
import os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import folium
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from torch.amp import autocast, GradScaler

# Set device for GPU acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler_amp = GradScaler(device='cuda')
print(f"🚀 Modeling environment ready. Using device: {device}")

/sfs/gpfs/tardis/home/upw4ys/Documents/floodnet_work/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [4]:
# 1. Load full dataset (Keep 'df' for your EDA)
FILE_PATH = "rain_influenced_gages.parquet"
df = pd.read_parquet(FILE_PATH)

# 2. Modeling Features
features = ['precip_1hr [inch]', 'precip_max_intensity [inch/hour]',
            'temp_2m [degF]', 'soil_moisture_05cm [m^3/m^3]', 'elevation [feet]']
target = 'depth_inches'

# 3. Clean specifically for these columns
df_clean = df[features + [target]].dropna().astype('float32')

# 4. Temporal Split (80/20)
split_idx = int(len(df_clean) * 0.8)
train_df = df_clean.iloc[:split_idx]
test_df = df_clean.iloc[split_idx:]

# 5. Scaling (Done on CPU)
scaler_X, scaler_y = StandardScaler(), StandardScaler()
X_train_cpu = torch.tensor(scaler_X.fit_transform(train_df[features]), dtype=torch.float32)
y_train_cpu = torch.tensor(scaler_y.fit_transform(train_df[[target]]), dtype=torch.float32)
X_test_cpu = torch.tensor(scaler_X.transform(test_df[features]), dtype=torch.float32)
y_test_raw_cpu = torch.tensor(test_df[target].values, dtype=torch.float32)

print(f"✅ Data Ready in CPU RAM. Rows: {len(df_clean):,}. Original 'df' preserved.")

✅ Data Ready in CPU RAM. Rows: 32,572,701. Original 'df' preserved.


In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.ln = nn.LayerNorm(size)
        self.fc = nn.Linear(size, size)
    def forward(self, x):
        return x + F.relu(self.fc(self.ln(x)))

class SotaANN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input = nn.Linear(input_size, hidden_size)
        self.res1 = ResidualBlock(hidden_size)
        self.res2 = ResidualBlock(hidden_size)
        self.output = nn.Linear(hidden_size, 1)
    def forward(self, x):
        x = F.relu(self.input(x))
        return self.output(self.res2(self.res1(x)))

class SotaAttentionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(hidden_size * 2, 1)
        self.fc = nn.Linear(hidden_size * 2, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        weights = F.softmax(self.attn(out), dim=1)
        context = torch.sum(out * weights, dim=1)
        return self.fc(context)

def calculate_nse_gpu(y_true, y_pred):
    return 1 - (torch.sum((y_true - y_pred)**2) / torch.sum((y_true - torch.mean(y_true))**2))

In [6]:
class HydroWindowDataset(Dataset):
    def __init__(self, data, targets, window_size):
        self.data, self.targets, self.window_size = data, targets, window_size
    def __len__(self):
        return len(self.data) - self.window_size
    def __getitem__(self, idx):
        return self.data[idx : idx + self.window_size], self.targets[idx + self.window_size]

# --- 1. Log-Reg Objective ---
def objective_log_reg(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 10.0, log=True)
    c = trial.suggest_float("log_shift", 1e-4, 0.5, log=True)
    y_tr_log = np.log(train_df[target] + c)
    model = Ridge(alpha=alpha).fit(train_df[features], y_tr_log)
    preds = np.exp(model.predict(test_df[features])) - c
    return 1 - (np.sum((test_df[target].values - preds)**2) / np.sum((test_df[target].values - np.mean(test_df[target].values))**2))

# --- 2. ANN Objective ---
def objective_ann(trial):
    h_size = trial.suggest_int("hidden_size", 128, 512)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = 100000
    loader = DataLoader(TensorDataset(X_train_cpu, y_train_cpu), batch_size=batch_size, shuffle=True, pin_memory=True)
    model = SotaANN(len(features), h_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(10):
        for bx, by in loader:
            bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
            optimizer.zero_grad()
            with autocast(device_type='cuda'): loss = nn.MSELoss()(model(bx), by)
            scaler_amp.scale(loss).backward(); scaler_amp.step(optimizer); scaler_amp.update()
    model.eval()
    with torch.no_grad():
        p = model(X_test_cpu.to(device))
        m, s = torch.tensor(scaler_y.mean_, device=device), torch.tensor(scaler_y.scale_, device=device)
        return calculate_nse_gpu(y_test_raw_cpu.to(device), (p * s + m).flatten()).item()

# --- 3. LSTM Objective ---
def objective_lstm(trial):
    window = trial.suggest_int("window_size", 30, 90, step=30)
    h_size = trial.suggest_int("hidden_size", 32, 128)
    loader = DataLoader(HydroWindowDataset(X_train_cpu, y_train_cpu, window), batch_size=32768, pin_memory=True)
    model = SotaAttentionLSTM(len(features), h_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for _ in range(2):
        for bx, by in loader:
            bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
            optimizer.zero_grad()
            with autocast(device_type='cuda'): loss = nn.MSELoss()(model(bx), by)
            scaler_amp.scale(loss).backward(); scaler_amp.step(optimizer); scaler_amp.update()
    model.eval()
    with torch.no_grad():
        test_loader = DataLoader(HydroWindowDataset(X_test_cpu, y_test_raw_cpu, window), batch_size=65536)
        all_p = torch.cat([model(tx.to(device)).cpu() for tx, _ in test_loader])
        m, s = torch.tensor(scaler_y.mean_), torch.tensor(scaler_y.scale_)
        return 1 - (torch.sum((y_test_raw_cpu[window:] - (all_p * s + m).flatten())**2) / torch.sum((y_test_raw_cpu[window:] - torch.mean(y_test_raw_cpu))**2)).item()

In [ ]:
print("📊 Starting Model Shootout (Baseline + SOTA Proposal)...")
study_lr = optuna.create_study(direction="maximize"); study_lr.optimize(objective_log_reg, n_trials=30)
study_ann = optuna.create_study(direction="maximize"); study_ann.optimize(objective_ann, n_trials=20)
study_lstm = optuna.create_study(direction="maximize"); study_lstm.optimize(objective_lstm, n_trials=10)

print(f"\n🏆 Final Results (NSE):\nLog-Reg: {study_lr.best_value:.4f}\nANN: {study_ann.best_value:.4f}\nLSTM: {study_lstm.best_value:.4f}")

[I 2026-04-08 11:09:01,942] A new study created in memory with name: no-name-8cf24dba-d3c6-4bc0-a9b8-cc392cc14518


📊 Starting Model Shootout (Baseline + SOTA Proposal)...


[I 2026-04-08 11:09:03,527] Trial 0 finished with value: -0.0223848819732666 and parameters: {'alpha': 0.0016063150204894104, 'log_shift': 0.051969303322108015}. Best is trial 0 with value: -0.0223848819732666.
[I 2026-04-08 11:09:05,049] Trial 1 finished with value: -0.026518821716308594 and parameters: {'alpha': 0.004051273469269759, 'log_shift': 0.00013656100091982004}. Best is trial 0 with value: -0.0223848819732666.
[I 2026-04-08 11:09:06,552] Trial 2 finished with value: -0.02639913558959961 and parameters: {'alpha': 5.016174649587542, 'log_shift': 0.0007644936053828587}. Best is trial 0 with value: -0.0223848819732666.
[I 2026-04-08 11:09:08,052] Trial 3 finished with value: -0.025840401649475098 and parameters: {'alpha': 0.8851755046059826, 'log_shift': 0.0050143095815334245}. Best is trial 0 with value: -0.0223848819732666.
[I 2026-04-08 11:09:09,550] Trial 4 finished with value: -0.026066064834594727 and parameters: {'alpha': 0.12974691389770565, 'log_shift': 0.00311150815217

In [ ]:
# Best Log-Reg
b_lr = study_lr.best_params
lr_final = Ridge(alpha=b_lr['alpha']).fit(train_df[features], np.log(train_df[target] + b_lr['log_shift']))
lr_p = np.exp(lr_final.predict(test_df[features])) - b_lr['log_shift']

# Best ANN
b_ann = study_ann.best_params
ann_final = SotaANN(len(features), b_ann['hidden_size']).to(device)
opt_ann = optim.Adam(ann_final.parameters(), lr=b_ann['lr'])
ann_final.train()
loader = DataLoader(TensorDataset(X_train_cpu, y_train_cpu), batch_size=65536, shuffle=True)
for _ in range(15):
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        opt_ann.zero_grad()
        with autocast(device_type='cuda'): loss = nn.MSELoss()(ann_final(bx), by)
        scaler_amp.scale(loss).backward(); scaler_amp.step(opt_ann); scaler_amp.update()

ann_final.eval()
with torch.no_grad():
    p_s = ann_final(X_test_cpu.to(device))
    ann_p = (p_s.cpu() * scaler_y.scale_ + scaler_y.mean_).numpy().flatten()

In [ ]:
plt.figure(figsize=(15, 7), dpi=150)
win = slice(-1000, -200) # Flood event zoom
plt.plot(test_df[target].values[win], label='Observed (FloodNet)', color='black', lw=2.5)
plt.plot(lr_p[win], label=f'Log-Reg (NSE: {study_lr.best_value:.3f})', color='#e67e22', ls='--')
plt.plot(ann_p[win], label=f'Res-ANN (NSE: {study_ann.best_value:.4f})', color='#9b59b6')
plt.title("SOTA Performance Comparison: Flood Event Analysis", fontsize=15)
plt.ylabel("Inches"); plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
sensor_list = df['deployment_id'].unique().tolist()
print(f"Total Sensors: {len(sensor_list)}")
print("Available IDs:", sensor_list)

In [ ]:
df[['latitude', 'longitude']].isna().sum()

In [ ]:
# 1. List all unique sensors
sensor_names = df['deployment_id'].unique().tolist()
print(f"✅ Total unique sensors found: {len(sensor_names)}")
print(f"Sample IDs: {sensor_names[:10]}")

# 2. Create the Spatial Map (NAD83 / New York Long Island Context)
def create_sensor_map(df):
    # Aggregate max depth per sensor for visual scaling
    map_data = df.groupby(['deployment_id', 'latitude', 'longitude'])['depth_inches'].max().reset_index()

    # Initialize NYC Map
    m = folium.Map(location=[40.7128, -74.0060], zoom_start=11, tiles='CartoDB Positron')

    for _, row in map_data.iterrows():
    # 1. Scale the radius: 1.5 pixels per inch of flood
    # 2. Set a 'floor' of 3 pixels so dry sensors are still visible
    # 3. Use a square root for 'area-based' scaling (more scientifically accurate)
        scaled_radius = row['depth_inches'] * 0.5

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=scaled_radius,
            popup=f"ID: {row['deployment_id']}<br>Max Depth: {row['depth_inches']:.2f} in",
            color='#e74c3c',
            fill=True,
            fill_opacity=0.5,
            weight=1
        ).add_to(m)
    return m

# Display map in Colab
nyc_map = create_sensor_map(df)
nyc_map

In [ ]:
def plot_individual_storms(df, sensor_id):
    # 1. Filter and sort sensor data
    sensor_data = df[df['deployment_id'] == sensor_id].sort_values('time').copy()

    if sensor_data.empty:
        print(f"❌ No data found for ID: {sensor_id}")
        return

    # 2. EVENT ID LOGIC: Group rows where time gap <= 120 minutes
    # This prevents the "400k events" error by keeping buffers together
    sensor_data['time_diff'] = sensor_data['time'].diff().dt.total_seconds() / 60
    sensor_data['event_id'] = (sensor_data['time_diff'] > 120).cumsum()

    unique_events = sensor_data['event_id'].unique()
    print(f"🔍 Found {len(unique_events)} discrete storm windows for {sensor_id}.")

    # 3. Plotting Loop
    for eid in unique_events:
        subset = sensor_data[sensor_data['event_id'] == eid]

        # Filtering out 'noise' (events shorter than 6 hours)
        if len(subset) < 360:
            continue

        fig, ax1 = plt.subplots(figsize=(14, 6), dpi=100)

        # Primary Axis: Water Depth (Black)
        color_depth = '#1a1a1a'
        ax1.set_xlabel('Time (UTC)', fontsize=10)
        ax1.set_ylabel('Water Depth (inches)', color=color_depth, fontsize=12, fontweight='bold')
        ax1.plot(subset['time'], subset['depth_inches'], color=color_depth, linewidth=2, label='Depth')
        ax1.fill_between(subset['time'], subset['depth_inches'], color=color_depth, alpha=0.1)
        ax1.tick_params(axis='y', labelcolor=color_depth)

        # Secondary Axis: Precipitation (Blue - Inverted)
        ax2 = ax1.twinx()
        color_rain = '#0072B2'
        ax2.set_ylabel('1-hr Precipitation (inches)', color=color_rain, fontsize=12, fontweight='bold')
        ax2.bar(subset['time'], subset['precip_1hr [inch]'], color=color_rain, width=0.005, alpha=0.6, label='Precip')

        # Scaling and Inversion
        max_rain = subset['precip_1hr [inch]'].max()
        ax2.set_ylim(max(max_rain * 4, 0.5), 0)
        ax2.tick_params(axis='y', labelcolor=color_rain)

        # Title and Formatting
        event_start = subset['time'].min().strftime('%Y-%m-%d')
        plt.title(f"Storm Event Analysis: {sensor_id} | Starting {event_start}\n"
                  f"(24h Antecedent & Recession Phases Included)", fontsize=14, pad=15)
        plt.grid(True, linestyle='--', alpha=0.3)

        fig.tight_layout()
        plt.show()

# --- INTERACTIVE EXECUTION ---
plot_individual_storms(df, "simply_half_monkey")

In [ ]:
def plot_journal_hydrograph(df, sensor_id):
    # Subset data for a single sensor's event
    subset = df[df['deployment_id'] == sensor_id].sort_values('time')

    fig, ax1 = plt.subplots(figsize=(14, 7), dpi=100)

    # --- Axis 1: Flood Depth ---
    color_depth = '#1a1a1a' # Dark grey/black for clarity
    ax1.set_xlabel('Time (UTC)', fontsize=12)
    ax1.set_ylabel('Water Depth (inches)', color=color_depth, fontsize=12, fontweight='bold')
    ax1.plot(subset['time'], subset['depth_inches'], color=color_depth, linewidth=2.5, label='Observed Depth')
    ax1.fill_between(subset['time'], subset['depth_inches'], color=color_depth, alpha=0.15)
    ax1.tick_params(axis='y', labelcolor=color_depth)

    # --- Axis 2: Precipitation (Inverted) ---
    ax2 = ax1.twinx()
    color_rain = '#0072B2' # Hydrology Blue
    ax2.set_ylabel('1-hr Precipitation (inches)', color=color_rain, fontsize=12, fontweight='bold')
    ax2.bar(subset['time'], subset['precip_1hr [inch]'], color=color_rain, width=0.005, alpha=0.7, label='Precipitation')

    # Invert Axis 2: High rain comes from the top
    ax2.set_ylim(subset['precip_1hr [inch]'].max() * 4, 0)
    ax2.tick_params(axis='y', labelcolor=color_rain)

    # Annotate the Buffers
    plt.title(f"Detailed Storm Event Analysis: {sensor_id}\nAntecedent & Recession Phases (24h Buffers)", fontsize=14, pad=20)
    plt.grid(True, which='both', linestyle='--', alpha=0.4)

    fig.tight_layout()
    plt.show()

plot_journal_hydrograph(df, df['deployment_id'].iloc[0])

In [ ]:
def plot_antecedent_correlation(df):
    # Group by sensor/event to find starting moisture vs peak depth
    # We define 'starting moisture' as the first record in the 24h window
    event_stats = df.groupby('deployment_id').agg({
        'soil_moisture_05cm [m^3/m^3]': 'first',
        'depth_inches': 'max'
    }).rename(columns={'depth_inches': 'peak_depth', 'soil_moisture_05cm [m^3/m^3]': 'initial_moisture'})

    plt.figure(figsize=(10, 6))
    sns.regplot(x='initial_moisture', y='peak_depth', data=event_stats,
                scatter_kws={'alpha':0.6, 'color':'#2c3e50'},
                line_kws={'color':'#e74c3c', 'lw':2})

    plt.title("Hypothesis Testing: Initial Soil Moisture vs. Peak Storm Depth", fontsize=13)
    plt.xlabel("Initial Soil Moisture at T-24h (m³/m³)", fontsize=11)
    plt.ylabel("Peak Flood Depth (inches)", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.show()

plot_antecedent_correlation(df)

In [ ]:
def plot_lag_correlation(df, sensor_id):
    sensor_data = df[df['deployment_id'] == sensor_id].copy()

    # Calculate correlations at different lags (in 5-min increments)
    lags = np.arange(0, 37) * 5 # 0 to 180 minutes
    corrs = []

    for l in lags:
        # Shift precip forward to see correlation with depth
        shifted_rain = sensor_data['precip_1hr [inch]'].shift(l // 1) # Simplified lag
        corrs.append(sensor_data['depth_inches'].corr(shifted_rain))

    plt.figure(figsize=(10, 4))
    plt.plot(lags, corrs, marker='o', linestyle='-', color='#d35400')
    plt.axvline(lags[np.argmax(corrs)], color='black', linestyle='--', label=f'Peak Lag: {lags[np.argmax(corrs)]} min')

    plt.title(f"Lag-Time Identification for LSTM Windowing: {sensor_id}", fontsize=13)
    plt.xlabel("Time Lag (Minutes)", fontsize=11)
    plt.ylabel("Pearson Correlation ($r$)", fontsize=11)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

plot_lag_correlation(df, df['deployment_id'].iloc[0])

In [ ]:
def plot_correlation_heatmap(df):
    # Select the numerical columns most likely to be used as model features
    features = [
        'elevation [feet]', 'temp_2m [degF]', 'relative_humidity [percent]',
        'dewpoint [degF]', 'precip_max_intensity [inch/hour]', 'precip_1hr [inch]',
        'soil_temp_05cm [degF]', 'soil_moisture_05cm [m^3/m^3]',
        'soil_moisture_25cm [m^3/m^3]', 'snow_depth [inch]'
    ]

    # Calculate the correlation matrix
    corr_matrix = df[features].corr(method='spearman')

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', linewidths=0.5)
    plt.title("Spearman Rank Correlation of Hydro-Meteorological Features", fontsize=14)
    plt.show()

plot_correlation_heatmap(df)

In [ ]:
def calculate_vif(df):
    # Select features and drop rows with NaNs (VIF requires clean data)
    features_to_test = [
        'temp_2m [degF]', 'relative_humidity [percent]', 'dewpoint [degF]',
        'soil_temp_05cm [degF]', 'soil_moisture_05cm [m^3/m^3]', 'elevation [feet]'
    ]

    X = df[features_to_test].dropna()
    X = add_constant(X) # Add intercept for VIF calculation

    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

    # Filter out the constant and sort
    return vif_data[vif_data["Feature"] != "const"].sort_values("VIF", ascending=False)

vif_results = calculate_vif(df)
print(vif_results)